<a href="https://colab.research.google.com/github/marcoastupp/ai-flux-training/blob/main/Treino_Flux_AI_Toolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import sys
from google.colab import drive

# 0. Mudar IMEDIATAMENTE para /content
os.chdir('/content')

# 1. Montar o Google Drive
drive.mount('/content/drive', force_remount=False)

# 2. Configurar o ambiente virtual Python 3.10 e isolar caminhos globais
!add-apt-repository ppa:deadsnakes/ppa -y > /dev/null 2>&1
!apt-get update > /dev/null 2>&1
!apt-get install python3.10 python3.10-dev python3.10-venv python3.10-distutils -y > /dev/null 2>&1

!python3.10 -m venv /content/venv

# Remover caminhos globais do Colab para evitar contaminação
sys.path = [p for p in sys.path if 'usr/local' not in p and 'python3.13' not in p and '/content/ai-toolkit' not in p]
import site
site.addsitedir('/content/venv/lib/python3.10/site-packages')

!/content/venv/bin/python -m pip install --upgrade pip > /dev/null 2>&1

# 3. Limpar e clonar o ai-toolkit
os.chdir('/content')
!rm -rf /content/ai-toolkit
!git clone https://github.com/ostris/ai-toolkit.git

# 4. Instalação estrita e limpa de dependências no venv
print("📦 Instalando dependências essenciais...")
!/content/venv/bin/python -m pip install -q --upgrade pip
!/content/venv/bin/python -m pip install -q numpy==1.26.4
!/content/venv/bin/python -m pip install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!/content/venv/bin/python -m pip install -q transformers==4.45.0 diffusers==0.30.0 accelerate==0.33.0
!/content/venv/bin/python -m pip install -q safetensors timm einops omegaconf sentencepiece peft==0.11.1 bitsandbytes==0.43.1 python-dotenv oyaml opencv-python-headless==4.9.0.80 albumentations av lpips lycoris-lora --no-deps
!/content/venv/bin/python -m pip uninstall -y triton > /dev/null 2>&1

# 5. Aplicar patches diretamente nos ficheiros dentro de /content/ai-toolkit
os.chdir('/content/ai-toolkit')

# Patch cirúrgico no toolkit/pipelines.py para evitar o erro do FluxControlPipeline
pipelines_path = "toolkit/pipelines.py"
if os.path.exists(pipelines_path):
    with open(pipelines_path, "r") as f:
        pipe_content = f.read()

    old_target_pipe = "FluxPipeline, FluxControlPipeline"
    safe_replacement_pipe = """FluxPipeline
try:
    from diffusers import FluxControlPipeline
except ImportError:
    class FluxControlPipeline:
        pass"""

    if old_target_pipe in pipe_content:
        pipe_content = pipe_content.replace(old_target_pipe, safe_replacement_pipe)
        with open(pipelines_path, "w") as f:
            f.write(pipe_content)

# Reescrever o stable_diffusion_model.py para evitar erros de importação
sd_model_path = "toolkit/stable_diffusion_model.py"
if os.path.exists(sd_model_path):
    with open(sd_model_path, "r") as f:
        sd_code = f.read()

    old_import_block = """from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionXLPipeline,
    T2IAdapter,
    DDPMScheduler,
    StableDiffusionXLImg2ImgPipeline,
    FluxPipeline,
    FluxControlPipeline,
    Lumina2Pipeline,
    FlowMatchEulerDiscreteScheduler,
    SD3Transformer2Model,
    Lumina2Transformer2DModel
)"""

    safe_import_block = """from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionXLPipeline,
    T2IAdapter,
    DDPMScheduler,
    StableDiffusionXLImg2ImgPipeline,
    FluxPipeline,
    FlowMatchEulerDiscreteScheduler,
    SD3Transformer2Model
)
try:
    from diffusers import FluxControlPipeline
except ImportError:
    FluxControlPipeline = None
try:
    from diffusers import Lumina2Pipeline
except ImportError:
    Lumina2Pipeline = None
try:
    from diffusers import Lumina2Transformer2DModel
except ImportError:
    Lumina2Transformer2DModel = None
"""
    if old_import_block in sd_code:
        sd_code = sd_code.replace(old_import_block, safe_import_block)
        with open(sd_model_path, "w") as f:
            f.write(sd_code)

# Patch para o lycoris
lycoris_library_path = "/content/venv/lib/python3.10/site-packages/lycoris/kernels/library.py"
if os.path.exists(lycoris_library_path):
    with open(lycoris_library_path, "r") as f:
        lyc_content = f.read()
    old_target_lyc = "@torch.library.custom_op"
    safe_replacement_lyc = """if not hasattr(torch.library, "custom_op"):
    class CustomOpMock:
        def __init__(self, func):
            self.func = func
        def __call__(self, *args, **kwargs):
            return self.func(*args, **kwargs)
        def register_fake(self, *args, **kwargs):
            def decorator(f): return f
            return decorator
    def custom_op(name, mutates_args=(), device_types=None):
        def decorator(func): return CustomOpMock(func)
        return decorator
    torch.library.custom_op = custom_op
@torch.library.custom_op"""
    if old_target_lyc in lyc_content and "CustomOpMock" not in lyc_content:
        lyc_content = lyc_content.replace(old_target_lyc, safe_replacement_lyc)
        with open(lycoris_library_path, "w") as f:
            f.write(lyc_content)

# Criar Stubs do torchao
torchao_mock_dir = "/content/venv/lib/python3.10/site-packages/torchao"
quant_dir = os.path.join(torchao_mock_dir, "quantization")
os.makedirs(quant_dir, exist_ok=True)
with open(os.path.join(torchao_mock_dir, "__init__.py"), "w") as f: f.write("# Mock\n")
with open(os.path.join(quant_dir, "__init__.py"), "w") as f: f.write("# Mock\n")
with open(os.path.join(quant_dir, "quant_primitives.py"), "w") as f: f.write("_DTYPE_TO_BIT_WIDTH = {}\n")
with open(os.path.join(quant_dir, "autoquant.py"), "w") as f: f.write("class autoquant: pass\nclass AutoQuantizableLinearWeight: pass\n")

# Stubs de adaptadores
with open("toolkit/lora_special.py", "w") as f: f.write("class LoRASpecialNetwork: pass\n")
with open("toolkit/models/i2v_adapter.py", "w") as f: f.write("class I2VAdapter: pass\n")

# 6. Preparar diretórios e dataset
os.chdir('/content')
dataset_dir = "/content/dataset_max"
output_dir = "/content/drive/MyDrive/ComfyUI/models/loras"
img_train_dir = "/content/ai-toolkit/input_images"

os.makedirs(dataset_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
os.makedirs(img_train_dir, exist_ok=True)

print("📂 Sincronizando imagens e legendas...")
!cp -r /content/drive/MyDrive/ComfyUI/Max_dataset/* {dataset_dir}/
!rm -rf {img_train_dir}/*
os.makedirs(f"{img_train_dir}/10_sargento_max_wolf", exist_ok=True)
!cp -r {dataset_dir}/* {img_train_dir}/10_sargento_max_wolf/

# 7. Criar ficheiro YAML de configuração
config_content = """job: extension
config:
  name: sargento_max_wolf_flux_toolkit
  process:
    - type: train_lora
      training_folder: /content/ai-toolkit/output
      device: cuda:0
      trigger_word: "sargento max wolf"
      network:
        type: lora
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 250
        max_step_saves_to_keep: 3
        push_to_hub: false
      weighting_scheme: sigma_sqrt
      datasets:
        - folder_path: /content/ai-toolkit/input_images
          caption_ext: txt
          caption_dropout_rate: 0.05
          shuffle_tokens: false
          cache_latents_to_disk: true
          resolution: [512, 768]
      train:
        batch_size: 1
        steps: 600
        gradient_accumulation_steps: 4
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: flowmatch
        optimizer: adamw8bit
        lr: 1e-4
        ema_config:
          use_ema: false
        dtype: bf16
      model:
        name_or_path: black-forest-labs/FLUX.1-schnell
        is_flux: true
        quantize: false
"""
with open("/content/ai-toolkit/config_max.yaml", "w") as f:
    f.write(config_content)

print("✅ Ambiente preparado e corrigido com sucesso!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'ai-toolkit'...
remote: Enumerating objects: 15056, done.
remote: Counting objects: 100% (7222/7222), done.
remote: Compressing objects: 100% (1101/1101), done.
remote: Total 15056 (delta 6513), reused 6121 (delta 6121), pack-reused 7834 (from 3)
Receiving objects: 100% (15056/15056), 35.22 MiB | 25.01 MiB/s, done.
Resolving deltas: 100% (10442/10442), done.
📦 Instalando dependências essenciais...
📂 Sincronizando imagens e legendas...
✅ Ambiente preparado e corrigido com sucesso!
